In [12]:
# Load geojson as dataframe
import geopandas as gpd

gdf = gpd.read_file("final/final_output.geojson")

In [14]:
import pandas as pd

gdf.head()

# Figure out quintiles to color by
# get the investments per capita (total_investment_dollars / total_population)
gdf["investment_dollars_per_capita"] = gdf["total_investment_dollars"] / gdf["total_population"]

# get the quintile breakpoints
quintile_breaks = gdf["investment_dollars_per_capita"].quantile([0.2, 0.4, 0.6, 0.8]).values

# assign quintile labels (1-5) to each row
gdf["investment_dollars_per_capita_quintile"] = pd.cut(
    gdf["investment_dollars_per_capita"],
    bins=[-float("inf")] + list(quintile_breaks) + [float("inf")],
    labels=[1, 2, 3, 4, 5]
)

# print the quintile breakpoints and head
print("Quintile breakpoints:", quintile_breaks)
print(gdf[["investment_dollars_per_capita", "investment_dollars_per_capita_quintile"]].head())


Quintile breakpoints: [ 45.30702375  81.05456271 121.6377597  191.98828354]
   investment_dollars_per_capita investment_dollars_per_capita_quintile
0                      61.341118                                      2
1                      31.978595                                      1
2                      83.196838                                      3
3                     108.439857                                      3
4                      77.374758                                      2


In [16]:
# Do kmeans clustering to get 5 clusters
from sklearn.cluster import KMeans

# Drop rows with NaN in 'investment_dollars_per_capita' before clustering
gdf_nonan = gdf.dropna(subset=["investment_dollars_per_capita"]).copy()

kmeans = KMeans(n_clusters=5, random_state=42)
gdf_nonan["cluster"] = kmeans.fit_predict(gdf_nonan[["investment_dollars_per_capita"]])

# Merge the cluster labels back into the original gdf (NaNs will remain for rows that were dropped)
gdf["cluster"] = gdf_nonan["cluster"]

# print the cluster labels and head
print("Cluster labels:", gdf["cluster"].dropna().unique())
print(gdf[["investment_dollars_per_capita", "cluster"]].head())

Cluster labels: [0. 2. 1. 4. 3.]
   investment_dollars_per_capita  cluster
0                      61.341118      0.0
1                      31.978595      0.0
2                      83.196838      0.0
3                     108.439857      2.0
4                      77.374758      0.0


In [7]:
# Run tippacanoe to generate pmtiles from geojson
# May need to cast ints to floats
import subprocess

cmd = [
  "tippecanoe",
  "-o",
  "502-investments-map/public/tiles.pmtiles",
  "-zg",
  "--drop-densest-as-needed",
  "--extend-zooms-if-still-dropping",
  "-l",
  "final_output",
  "final/final_output.geojson"
]

subprocess.run(cmd, check=True)

3807 features, 2084305 bytes of geometry and attributes, 175500 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
Choosing a maxzoom of -z1 for features typically 148511 feet (45267 meters) apart, and at least 62780 feet (19136 meters) apart
Choosing a maxzoom of -z5 for resolution of about 9014 feet (2747 meters) within features
  99.9%  5/8/12  


CompletedProcess(args=['tippecanoe', '-o', '502-investments-map/public/tiles.pmtiles', '-zg', '--drop-densest-as-needed', '--extend-zooms-if-still-dropping', '-l', 'final_output', 'final/final_output.geojson'], returncode=0)